<style>
    img {
        display: block;
        margin-left: auto;
        margin-right: auto;
    }
</style>

<div style="background-color:#000047; padding: 30px; border-radius: 10px; color: white; text-align: center;">
    <img src='Figures/alinco_white_text.png' style="height: 100px; margin-bottom: 10px;"/>
    <h1> Módulo 3: Introducción al Aprendizaje por Refuerzo (Reinforcement Learning)</h1>
    <h3>Aprendizaje Automático Avanzado 2026</h3>
</div>

## 1. ¿Qué es el aprendizaje por refuerzo?

El **aprendizaje por refuerzo** (*Reinforcement Learning*, RL) es un paradigma de aprendizaje automático en el que un **agente** aprende a tomar decisiones mediante la interacción con un **entorno**. En cada paso, el agente recibe una recompensa que le indica qué tan conveniente fue su decisión. El objetivo no es acertar una etiqueta conocida, sino aprender una estrategia que maximice la recompensa acumulada a lo largo del tiempo.

RL es una de las tres grandes ramas del aprendizaje automático:

| Paradigma | Tipo de retroalimentación | Objetivo principal | Ejemplo típico |
| :--- | :--- | :--- | :--- |
| **Supervisado** | Etiquetas conocidas ($y$) proporcionadas por un supervisor. | Aprender una función que aproxime $f(x) \approx y$. | Clasificación de imágenes o detección de spam. |
| **No supervisado** | No hay etiquetas; se observan principalmente los datos de entrada ($X$). | Encontrar patrones, agrupamientos o representaciones latentes. | *Clustering* con K-Means o GMM, y reducción de dimensionalidad con PCA. |
| **Por refuerzo (RL)** | Señal escalar de recompensa ($R_t$) generada durante la interacción con el entorno. | Maximizar la recompensa acumulada a largo plazo, llamada **retorno**. | Navegación autónoma, robótica y juegos como ajedrez, Go o Atari. |

La diferencia central es que el agente debe aprender **qué hacer**, **cuándo hacerlo** y **cómo valorar las consecuencias futuras** de sus decisiones.

## 2. Elementos del aprendizaje por refuerzo y ciclo de interacción

El aprendizaje por refuerzo se describe mediante dos entidades que interactúan de forma continua:

1. **Agente (*agent*)**: entidad que observa el estado, toma decisiones y aprende una estrategia de comportamiento.
2. **Entorno (*environment*)**: sistema externo con el que interactúa el agente y que determina las transiciones y las recompensas.

<img src="Figures/mdp_diagram.svg" alt="Diagrama MDP Agente-Entorno" style="display: block; margin: 0 auto;" width="450">

### Ciclo temporal discreto

En cada instante $t = 0, 1, 2, \dots$ ocurre lo siguiente:

1. El agente observa el **estado** actual $S_t \in \mathcal{S}$.
2. Con base en ese estado y en su política, selecciona una **acción** $A_t \in \mathcal{A}(S_t)$.
3. El entorno ejecuta la acción y transita a un nuevo estado $S_{t+1} \in \mathcal{S}$.
4. El entorno emite una **recompensa** escalar $R_{t+1} \in \mathbb{R}$, que evalúa la consecuencia inmediata de la acción.

La **política** es la regla que determina cómo el agente elige sus acciones. Puede entenderse como el modelo de decisión del agente: a medida que este interactúa con el entorno, la política se ajusta para favorecer decisiones que produzcan mejores recompensas acumuladas.

El proceso puede resumirse así:

1. **Acción**: el agente selecciona una acción a partir del estado $S_t$ y de su política.
2. **Recompensa**: después de ejecutar la acción, el entorno devuelve una recompensa que puede ser positiva, negativa o nula.
3. **Nuevo estado**: la acción provoca una transición de $S_t$ a $S_{t+1}$.
4. **Actualización**: el agente utiliza la experiencia obtenida para mejorar su política o sus estimaciones de valor.

<img src="./Figures/001_RL.png" alt="Esquema del aprendizaje por refuerzo" style="width: 600px;"/>

### 2.1. Terminología básica

| Elemento | Descripción |
|---|---|
| **Agente** | Entidad que vive en un entorno, toma decisiones y aprende a partir de sus consecuencias. |
| **Entorno** | Sistema que el agente puede observar y sobre el que puede actuar. |
| **Estado** | Representación de la situación actual del entorno relevante para tomar una decisión. |
| **Acción** | Una de las opciones disponibles para el agente en un estado determinado. |
| **Recompensa** | Señal numérica que indica qué tan conveniente fue una transición o una acción. |
| **Política** | Regla, determinista o probabilística, que relaciona estados con acciones. |
| **Función de valor** | Función que estima qué tan conveniente es encontrarse en un estado, considerando las recompensas futuras. |
| **Q-Table** | Tabla que asigna un valor a cada pareja estado-acción. Sus valores ayudan a elegir acciones, pero la política se obtiene a partir de ellos. |

Una Q-Table sencilla puede representarse así:

| Estado | Acción | Valor |
|---|---|---|
| $(0,0)$ | Arriba | 0,0 |
| $(0,0)$ | Abajo | 0,5 |
| $(0,0)$ | Izquierda | 0,0 |
| $(0,0)$ | Derecha | 0,6 |
| $(0,1)$ | Arriba | 0,0 |
| $(0,1)$ | Abajo | 2,3 |
| $(0,1)$ | Izquierda | 0,1 |
| $(0,1)$ | Derecha | 4,7 |
| ... | ... | ... |
| $(n,m)$ | Arriba | 3,6 |
| $(n,m)$ | Abajo | 0,0 |
| $(n,m)$ | Izquierda | 7,2 |
| $(n,m)$ | Derecha | 0,0 |

### 2.2. Ejemplo intuitivo: aprender el camino a casa

En una tarea secuencial del mundo real, como conducir un vehículo autónomo o controlar un robot, no existe un supervisor que indique exactamente qué acción ejecutar en cada instante. El agente debe tomar decisiones, observar sus consecuencias y aprender mediante ensayo y error.

Supongamos que debemos aprender a ir del trabajo a casa sin utilizar mapas ni GPS. En esta analogía:

- **Agente**: la persona.
- **Acciones**: las decisiones sobre qué calle tomar.
- **Entorno**: la ciudad.
- **Recompensa**: llegar a casa; cuanto menos tiempo se tarde, mejor será la recompensa.

<img src="./Figures/002_RL.png" alt="Ejemplo de rutas: primer recorrido" style="width: 200px;"/>

El primer día no tenemos conocimiento previo. En cada intersección probamos una calle hasta encontrar una ruta que nos permita llegar a casa. La ruta aprendida puede ser larga, pero ya proporciona información útil.

<img src="./Figures/003_RL.png" alt="Ejemplo de rutas: segundo recorrido" style="width: 200px;"/>

En los días siguientes podemos reutilizar la ruta conocida o probar alternativas. Así combinamos el conocimiento adquirido con nuevas decisiones que podrían revelar un camino mejor.

<img src="./Figures/004_RL.png" alt="Ejemplo de rutas: aprendizaje de nuevas alternativas" style="width: 200px;"/>

Después de repetir la tarea muchas veces, probablemente encontraremos una ruta corta. No necesariamente será la mejor ruta posible, por lo que aún puede ser conveniente explorar de vez en cuando.

<img src="./Figures/005_RL.png" alt="Ejemplo de rutas: ruta aprendida" style="width: 200px;"/>

Este ejemplo muestra la idea central del aprendizaje por refuerzo: el agente realiza acciones en un entorno, recibe recompensas y modifica su política para mejorar el resultado a largo plazo.

### 2.3. Exploración y explotación

**Exploración** significa probar acciones poco conocidas o aleatorias para obtener información sobre el entorno. En el ejemplo, ocurre cuando probamos calles nuevas porque todavía no sabemos cuál conduce de manera más rápida a casa.

**Explotación** significa elegir la mejor acción de acuerdo con el conocimiento disponible. En el ejemplo, ocurre cuando seguimos la ruta que hasta el momento ha producido el mejor resultado.

Al inicio de una tarea suele ser necesario explorar porque el agente tiene poca información. Conforme aprende, puede explotar más el conocimiento adquirido. Sin embargo, una política que solo explota puede quedar atrapada en una solución aceptable y no descubrir una alternativa mejor. Por eso, muchos algoritmos mantienen una pequeña probabilidad de exploración.

<img src="./Figures/006_RL.png" alt="Comparación entre rutas exploradas" style="width: 200px;"/>

La proporción entre exploración y explotación es un aspecto central del diseño de algoritmos de aprendizaje por refuerzo y se retomará más adelante con la estrategia $\epsilon$-greedy.


## 3. Proceso de decisión de Markov (MDP)

El **proceso de decisión de Markov** (*Markov Decision Process*, MDP) es el marco formal que sustenta gran parte del aprendizaje por refuerzo.

### 3.1. Propiedad de Markov

Un proceso cumple la **propiedad de Markov** cuando la distribución del siguiente estado y de la recompensa depende del estado y de la acción actuales, pero no requiere conocer toda la historia anterior. En otras palabras, el estado actual resume la información relevante del pasado para tomar la siguiente decisión.

En un MDP, esta propiedad se expresa como:

$$P(S_{t+1} = s_{t+1}, R_{t+1} = r_{t+1} \mid S_t = s_t, A_t = a_t, S_{t-1}, A_{t-1}, \dots, S_0, A_0)$$
$$= P(S_{t+1} = s_{t+1}, R_{t+1} = r_{t+1} \mid S_t = s_t, A_t = a_t)$$

> El futuro es condicionalmente independiente del pasado dado el estado presente. Por ello, una representación adecuada del estado debe contener la información necesaria para predecir las consecuencias de las acciones.

### 3.2. Definición formal

Un MDP se define mediante la 5-tupla $\left(\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma\right)$:

1. $\mathcal{S}$: conjunto de estados válidos, o **espacio de estados**.
2. $\mathcal{A}$: conjunto de acciones posibles, o **espacio de acciones**.
3. $\mathcal{P}$: función de probabilidad de transición:
   $$\mathcal{P}(s', r \mid s, a) = P(S_{t+1} = s', R_{t+1} = r \mid S_t = s, A_t = a)$$
4. $\mathcal{R}$: recompensa esperada:
   $$\mathcal{R}(s, a) = \mathbb{E}[R_{t+1} \mid S_t = s, A_t = a]$$
5. $\gamma \in [0, 1)$: **factor de descuento**, que controla la importancia relativa de las recompensas futuras frente a las inmediatas.

La formulación MDP permite pasar de la intuición del ejemplo de las rutas a una descripción matemática de las decisiones, las transiciones y las recompensas.

## 4. Ecuación de Bellman y propagación del valor

La **ecuación de Bellman** expresa una idea fundamental: el valor de una decisión es la recompensa inmediata más el valor descontado de las decisiones futuras. Esta relación permite resolver problemas secuenciales dividiéndolos en subproblemas más pequeños.

La programación dinámica puede aplicarse cuando el problema presenta:

- **Subestructura óptima**: una solución óptima puede construirse a partir de soluciones óptimas de subproblemas.
- **Subproblemas superpuestos**: los mismos subproblemas aparecen varias veces y sus resultados pueden reutilizarse.

Una forma sencilla de la ecuación de Bellman para el valor de un estado es:

$$V(s_t) = \underset{a}{\max} \left(R(s_t,a) + \gamma V(s_{t+1})\right)$$

Aquí, $R(s_t,a)$ es la recompensa por tomar la acción $a$ desde $s_t$, $V(s_{t+1})$ es el valor del estado siguiente y $\gamma$ es el factor de descuento.

### 4.1. Ejemplo de propagación del valor

Consideremos el siguiente tablero. El objetivo es alcanzar el estado $[4,4]$, que proporciona una recompensa de 100 puntos, evitando el estado $[4,3]$, que proporciona una penalización de -100 puntos.

<img src="./Figures/007_RL.png" alt="Tablero con recompensas terminales" style="width: 300px;"/>

Al comenzar desde $[1,1]$, los valores de los estados aún no se conocen y pueden inicializarse en cero. Después de explorar, la información de la recompensa comienza a propagarse hacia los estados vecinos.

<img src="./Figures/008_RL.png" alt="Primera iteración de propagación del valor" style="width: 800px;"/>

En una segunda iteración, la información alcanza nuevos estados:

<img src="./Figures/009_RL.png" alt="Segunda iteración de propagación del valor" style="width: 800px;"/>

Después de varias iteraciones, el valor se propaga por todo el tablero:

<img src="./Figures/010_RL.png" alt="Valores propagados en el tablero" style="width: 300px;"/>

Una vez estimado el valor de los estados, una estrategia posible consiste en seleccionar, en cada paso, el estado adyacente con mayor valor. Así se construye una secuencia de acciones que conduce hacia la recompensa máxima.

<img src="./Figures/011_RL.png" alt="Estrategias para alcanzar la recompensa" style="width: 300px;"/>

### 4.2. Recompensas parciales y penalización por paso

Si solo otorgamos una recompensa al llegar al destino, una ruta de seis pasos y otra de infinitos pasos podrían recibir el mismo valor final. Para favorecer rutas más cortas, podemos incluir una penalización por cada acción:

$$V(s_t) = \underset{a}{\max} \left(R(s_t,a) - \text{Penalty} + \gamma V(s_{t+1})\right)$$

Por ejemplo, si la penalización por paso es $-1$, cada movimiento reduce ligeramente el valor acumulado y el agente aprende a preferir soluciones que lleguen al objetivo en menos pasos.

<img src="./Figures/012_RL.png" alt="Valores con penalización por paso" style="width: 800px;"/>

## 5. Q-Function y Q-Table

La **Q-Function**, o función de valor de acción, estima qué tan conveniente es ejecutar la acción $a$ en el estado $s$ y continuar después con la mejor estrategia disponible:

$$Q(s_t, a) = R(s_t,a) + \gamma \underset{a'}{\max} Q(s_{t+1},a')$$

La diferencia entre el valor estimado antes y después de una transición se denomina **error de diferencia temporal** (*Temporal-Difference Error*):

$$TD(a,s) = \left(R(s_t,a) + \gamma \underset{a'}{\max} Q(s_{t+1},a')\right) - Q(s_t,a)$$

El valor se actualiza gradualmente mediante una tasa de aprendizaje $\alpha$:

$$\widehat{Q}(s, a) = Q(s, a) + \alpha \cdot TD(a,s)$$

Al combinar ambas expresiones obtenemos la actualización de Q-Learning:

$$\widehat{Q}(s, a) = Q(s, a) + \alpha \left[R(s,a) + \gamma \underset{a'}{\max} Q(s',a') - Q(s,a)\right]$$

La **Q-Table** almacena estos valores para cada pareja estado-acción. Al inicio puede inicializarse en cero y actualizarse conforme el agente explora el entorno.

| Estado | Acción 1 | Acción 2 | Acción 3 | Acción 4 |
|---|---|---|---|---|
| $(0,0)$ | 0,0 | 0,5 | 0,0 | 0,6 |
| $(0,1)$ | 0,0 | 2,3 | 0,1 | 4,7 |
| $(1,0)$ | 0,1 | 6,4 | 0,0 | 8,9 |
| ... | ... | ... | ... | ... |
| $(n,m)$ | 3,6 | 0,0 | 7,2 | 0,0 |

La Q-Table no es la política en sí misma: es una representación de los valores que la política puede utilizar para seleccionar acciones. En problemas grandes, esta tabla puede sustituirse por una función aproximadora, como una red neuronal.

## 6. Algoritmos básicos

Algunos algoritmos de aprendizaje por refuerzo son Q-Learning, SARSA, Deep Q-Learning, A3C y Monte Carlo. A continuación se presentan dos algoritmos tabulares fundamentales: Q-Learning y SARSA.

### 6.1. Q-Learning

Q-Learning es un algoritmo de diferencias temporales **fuera de política** (*off-policy*) y libre de modelo (*model-free*). Actualiza los valores utilizando la mejor acción posible en el siguiente estado, aunque esa acción no sea la que finalmente se ejecute durante la exploración.

$$\widehat{Q}(s, a) = Q(s, a) + \alpha \left[R(s,a) + \gamma \underset{a'}{\max} Q(s',a') - Q(s,a)\right]$$

- $\alpha \in (0,1]$: tasa de aprendizaje.
- $\gamma \in [0,1)$: factor de descuento.
- $\max_{a'}Q(s',a')$: valor de la mejor acción disponible en el siguiente estado.

En cada episodio, el agente parte del estado inicial, selecciona acciones, actualiza la Q-Table y termina cuando alcanza un estado terminal. La selección de acciones suele combinar:

- **Explotación**: elegir $\arg\max_a Q(s,a)$.
- **Exploración**: elegir una acción aleatoria.

La estrategia $\epsilon$-greedy controla este equilibrio. Si $\epsilon=0.1$, aproximadamente el 10 % de las decisiones se dedica a explorar y el 90 % a explotar el conocimiento adquirido.

<img src="./Figures/013_qlearning.png" alt="Pseudocódigo de Q-Learning" style="width: 500px;"/>

### 6.2. SARSA

SARSA (*State-Action-Reward-State-Action*) es similar a Q-Learning, pero actualiza el valor utilizando la acción que el agente realmente seleccionará en el siguiente estado. Por ello se considera un algoritmo **en política** (*on-policy*).

$$\widehat{Q}(s, a) = Q(s, a) + \alpha \left[R(s,a) + \gamma Q(s',a') - Q(s,a)\right]$$

En esta expresión, $a'$ es la acción elegida por la política en el estado siguiente $s'$. La diferencia esencial es:

- **Q-Learning**: utiliza $\max_{a'} Q(s',a')$, es decir, la mejor acción posible.
- **SARSA**: utiliza $Q(s',a')$, es decir, la acción que la política realmente tomaría.

<img src="./Figures/014_sarsa.png" alt="Pseudocódigo de SARSA" style="width: 500px;"/>

La elección entre ambos algoritmos depende del problema y de la política de exploración. SARSA puede considerar explícitamente el riesgo asociado a las acciones exploratorias, mientras que Q-Learning busca aprender el valor de la política óptima.

## 7. Formalización del retorno y de las funciones de valor

Las secciones anteriores introdujeron la actualización de Q-Learning y SARSA. Para completar la formalización, conviene precisar cómo se define el retorno y cómo se relacionan las funciones de valor con una política.

### 7.1. Retorno acumulado descontado

El objetivo del agente es maximizar el retorno esperado a partir del instante $t$:

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

- Si $\gamma = 0$, el agente solo considera la recompensa inmediata.
- Cuando $\gamma$ se aproxima a 1, el agente valora las recompensas que recibirá en un horizonte temporal más amplio.

### 7.2. Política

Una política $\pi$ define cómo selecciona acciones el agente a partir de los estados. Puede ser determinista o probabilística:

$$\pi(a \mid s) = P(A_t = a \mid S_t = s)$$

### 7.3. Funciones de valor

La **función de valor de estado** mide el retorno esperado al comenzar en el estado $s$ y seguir la política $\pi$:

$$V^\pi(s) = \mathbb{E}_\pi\left[G_t \mid S_t = s\right]$$

La **función de valor de acción** mide el retorno esperado al tomar primero la acción $a$ en el estado $s$ y seguir después la política $\pi$:

$$Q^\pi(s,a) = \mathbb{E}_\pi\left[G_t \mid S_t = s, A_t = a\right]$$

### 7.4. Ecuación de optimalidad de Bellman para $Q^*(s,a)$

Para la política óptima, la ecuación de Bellman para la función de acción es:

$$Q^*(s,a) = \mathcal{R}(s,a) + \gamma \sum_{s'} \mathcal{P}(s' \mid s,a) \max_{a'} Q^*(s',a')$$

Esta relación descompone el valor de una decisión en dos partes: la recompensa inmediata y el valor óptimo descontado de los estados que pueden alcanzarse después. Las actualizaciones tabulares de Q-Learning constituyen una forma iterativa de aproximarse a esta solución sin conocer explícitamente el modelo de transición.